### TinyML Model

Imagine you have a small sensor, like one you'd find in a smart home or a machine, that measures things like temperature and humidity. This notebook shows you how to:

1.  **Make Practice Data**: First, create fake sensor readings (temperature and humidity) and tag them as 'safe', 'warning', or 'critical'.

2.  **Teach a 'Brain'**: Then, use this practice data to train a simple Artificial Intelligence (AI) model, like teaching a small 'brain' to recognize patterns.

3.  **Translate to Tiny Computer Talk**: Next, take this AI 'brain' (which was built in a programming language like Python) and translate its knowledge into a very basic computer language (C code). This special code can then run on tiny, simple computers often found in sensors or smart devices (like an ESP32 chip), which don't have a lot of power or memory.
4.  **Fix for Tiny Computers**: Sometimes, the translated code needs a little tweak to work perfectly on these small devices. Add a small fix to make sure it runs without problems.
5.  **Check the 'Brain's' Work**: Finally, look inside our AI 'brain' to understand how it makes decisions. See which measurements (temperature or humidity) it cares about most, and test it with new readings to make sure it gives the right answers.

**The Big Idea**: The main goal is to create a smart program that can classify sensor data (`model.h` in C code) that's small and efficient enough to fit directly onto a low-power sensor device. This way, the sensor can make smart decisions right where the data is collected, without needing to send everything to a bigger computer, saving power and making it faster.

In [4]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000

# For simulating sensor readings
temperature = np.concatenate([
    np.random.uniform(20, 45, 600),   # SAFE range
    np.random.uniform(45, 60, 250),   # WARNING range
    np.random.uniform(60, 85, 150)    # CRITICAL range
])

humidity = np.random.uniform(30, 90, n)

# Create labels based on temperature
def label(t):
    if t < 45:
        return 0   # SAFE
    elif t < 60:
        return 1   # WARNING
    else:
        return 2   # CRITICAL

labels = np.array([label(t) for t in temperature])

# Save as CSV
df = pd.DataFrame({
    'temperature': temperature,
    'humidity': humidity,
    'label': labels
})

df.to_csv('thermal_data.csv', index=False)
print(df['label'].value_counts())
print(df.head())

label
0    600
1    250
2    150
Name: count, dtype: int64
   temperature   humidity  label
0    29.363503  41.107976      0
1    43.767858  62.514057      0
2    38.299849  82.376750      0
3    34.966462  73.933493      0
4    23.900466  78.393669      0




*   **Situation**: Needed a synthetic dataset for IoT sensor readings (temperature, humidity) to classify into 'SAFE', 'WARNING', 'CRITICAL'.

*   **Task**: Create a Pandas DataFrame with 1000 data points (temperature, humidity, label) and save it as `thermal_data.csv`.

*   **Action**: Generated random temperature and humidity values, defined labels based on temperature thresholds, combined into a DataFrame, and saved as `thermal_data.csv`.

*   **Result**: `thermal_data.csv` created with 1000 synthetic sensor readings, including 600 SAFE, 250 WARNING, and 150 CRITICAL labels, as confirmed by console output.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd

df = pd.read_csv('thermal_data.csv')

X = df[['temperature', 'humidity']]
y = df['label']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Evaluate
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       118
           1       1.00      1.00      1.00        51
           2       1.00      1.00      1.00        31

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200





**Machine learning model training and evaluation**. It performs the following steps:

*   **Loads Data**: Reads the `thermal_data.csv` file into a Pandas DataFrame.
*   **Prepares Data**: Splits the DataFrame into features (temperature, humidity) and the target variable (label).
*   **Splits Data**: Divides the dataset into training and testing sets to ensure robust model evaluation.
*   **Trains Model**: Initializes and trains a `RandomForestClassifier` on the training data.
*   **Evaluates Model**: Predicts labels for the test set and prints a `classification_report` to show performance metrics like precision, recall, and f1-score.

In [6]:
!pip install m2cgen

In [7]:
import m2cgen as m2c

# Convert model to pure C code (no libraries needed on ESP32)
code = m2c.export_to_c(model)

# Save to file
with open('model.h', 'w') as f:
    f.write(code)

print("Done! First 500 characters:")
print(code[:500])

Done! First 500 characters:
#include <string.h>
void add_vectors(double *v1, double *v2, int size, double *result) {
    for(int i = 0; i < size; ++i)
        result[i] = v1[i] + v2[i];
}
void mul_vector_number(double *v1, double num, int size, double *result) {
    for(int i = 0; i < size; ++i)
        result[i] = v1[i] * num;
}
void score(double * input, double * output) {
    double var0[3];
    double var1[3];
    double var2[3];
    double var3[3];
    double var4[3];
    double var5[3];
    double var6[3];
    double


Converting the trained machine learning model into C code for deployment on embedded systems:

1.  **Import `m2cgen`**: It imports the `m2cgen` library, which is specialized for transpiling machine learning models into native code.
2.  **Convert Model to C Code**: It uses `m2cgen.export_to_c(model)` to translate the previously trained `RandomForestClassifier` (`model`) into pure C code.
3.  **Save and Preview**: The generated C code is then saved into a file named `model.h`, and the first 500 characters of this code are printed to the console for a quick preview.

In [8]:
# Extract the decision thresholds directly from the trained model
for i, tree in enumerate(model.estimators_[:3]):
    print(f"Tree {i}: max_depth={tree.get_depth()}")

# Get feature importances
print("\nFeature importances:")
for name, imp in zip(['temperature', 'humidity'], model.feature_importances_):
    print(f"  {name}: {imp:.4f}")

# Test predictions
import numpy as np
import pandas as pd
test_cases = [[25, 50], [50, 60], [70, 45]]
for t, h in test_cases:
    pred = model.predict(pd.DataFrame([[t, h]], columns=['temperature', 'humidity']))[0]
    labels = ['SAFE', 'WARNING', 'CRITICAL']
    print(f"Temp={t}, Humidity={h} → {labels[pred]}")

Tree 0: max_depth=5
Tree 1: max_depth=2
Tree 2: max_depth=5

Feature importances:
  temperature: 0.9823
  humidity: 0.0177
Temp=25, Humidity=50 → SAFE
Temp=50, Humidity=60 → WARNING
Temp=70, Humidity=45 → CRITICAL



The trained `RandomForestClassifier` model and its prediction capabilities:

1.  **Analyze Decision Trees**: It prints the maximum depth of the first three decision trees within the model.
2.  **Evaluate Feature Importance**: It calculates and prints the `feature_importances_`, showing 'temperature' as significantly more influential than 'humidity'.
3.  **Test Model Predictions**: It defines sample test cases and uses the trained model to predict the `label` (SAFE, WARNING, CRITICAL) for each, printing the results.